In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor


# ---------- paths ----------
ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed_data"
RESULTS_DIR = ROOT / "results" / "intermediate"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DONOR_FILE = PROCESSED_DIR / "matching_geographic_distance.csv"
DETAIL_FILE = RESULTS_DIR / "Table2_G_XGBoost_detail.csv"
SUMMARY_FILE = RESULTS_DIR / "Table2_G_XGBoost_summary.csv"


# ---------- config ----------
TARGET = "power"
BASE_FEATURES = [
    "wind_speed",
    "temperature",
    "turbulence_intensity",
    "std_wind_direction",
]
ANGLE_FEATURE = "wind_direction"

K = 7
SEED = 2026
TRAIN_YEAR = 2017
TEST_YEARS = [2017, 2018]


# ---------- utils ----------
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if not np.any(m):
        return np.nan
    return float(np.sqrt(np.mean((y_true[m] - y_pred[m]) ** 2)))


def load_turbine_year(turbine_id: int, year: int) -> pd.DataFrame:
    path = DATA_DIR / f"Turbine{int(turbine_id)}_{int(year)}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(path)
    required = BASE_FEATURES + [ANGLE_FEATURE, TARGET]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")

    df = df[required].copy()
    for c in required:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna().copy()
    if df.empty:
        raise ValueError(f"No usable rows in {path}")

    rad = np.deg2rad(df[ANGLE_FEATURE].to_numpy())
    df["wind_direction_sin"] = np.sin(rad)
    df["wind_direction_cos"] = np.cos(rad)
    df = df.drop(columns=[ANGLE_FEATURE])

    return df


def feature_names():
    return BASE_FEATURES + ["wind_direction_sin", "wind_direction_cos"]


def read_geo_donor_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing donor file: {path}")

    df = pd.read_csv(path)
    required = {"target", "donor", "geo_distance"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {path}: {sorted(missing)}")

    df["target"] = pd.to_numeric(df["target"], errors="coerce")
    df["donor"] = pd.to_numeric(df["donor"], errors="coerce")
    df["geo_distance"] = pd.to_numeric(df["geo_distance"], errors="coerce")
    df = df.dropna(subset=["target", "donor", "geo_distance"]).copy()

    df["target"] = df["target"].astype(int)
    df["donor"] = df["donor"].astype(int)

    df = df.sort_values(["target", "geo_distance", "donor"]).reset_index(drop=True)
    return df


def top_k_geo_donors(geo_df: pd.DataFrame, target_id: int, k: int):
    sub = geo_df.loc[geo_df["target"] == target_id].copy()
    sub = sub.loc[sub["donor"] != target_id]
    sub = sub.sort_values(["geo_distance", "donor"])
    return sub["donor"].head(k).astype(int).tolist()


def build_model(seed: int):
    return LGBMRegressor(
        objective="regression",
        n_estimators=200,
        learning_rate=0.1,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        n_jobs=-1,
    )


def build_concat_training_data(donor_ids):
    feats = feature_names()
    frames = []

    for donor_id in donor_ids:
        try:
            df = load_turbine_year(donor_id, TRAIN_YEAR)
        except Exception:
            continue
        frames.append(df)

    if not frames:
        return None, None

    train_df = pd.concat(frames, axis=0, ignore_index=True)
    x_train = train_df[feats].to_numpy()
    y_train = train_df[TARGET].to_numpy()
    return x_train, y_train


def fit_concat_geo_model(donor_ids, target_id, test_year):
    feats = feature_names()

    x_train, y_train = build_concat_training_data(donor_ids)
    if x_train is None:
        raise ValueError(f"No usable donor data for target {target_id}")

    test_df = load_turbine_year(target_id, test_year)
    x_test = test_df[feats].to_numpy()
    y_test = test_df[TARGET].to_numpy()

    model = build_model(seed=target_id)

    start = time.time()
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    runtime = time.time() - start

    return {
        "pred": pred,
        "actual": y_test,
        "rmse": rmse(y_test, pred),
        "runtime": runtime,
    }


def run():
    geo_df = read_geo_donor_table(DONOR_FILE)
    targets = sorted(geo_df["target"].unique())

    detail_rows = []

    for target_id in targets:
        donors = top_k_geo_donors(geo_df, target_id, K)
        if not donors:
            continue

        for test_year in TEST_YEARS:
            try:
                res = fit_concat_geo_model(donors, target_id, test_year)
            except Exception:
                continue

            detail_rows.append(
                {
                    "method": "G_XGBoost",
                    "target": target_id,
                    "year": test_year,
                    "donors_used": ",".join(map(str, donors)),
                    "n_models": len(donors),
                    "rmse": res["rmse"],
                    "runtime_sec": res["runtime"],
                }
            )

    detail_df = pd.DataFrame(detail_rows)
    detail_df.to_csv(DETAIL_FILE, index=False)

    if detail_df.empty:
        summary_df = pd.DataFrame(
            columns=["method", "year", "avg_rmse", "total_runtime_sec"]
        )
    else:
        summary_df = (
            detail_df.groupby(["method", "year"], as_index=False)
            .agg(
                avg_rmse=("rmse", "mean"),
                total_runtime_sec=("runtime_sec", "sum"),
            )
        )

    summary_df.to_csv(SUMMARY_FILE, index=False)


if __name__ == "__main__":
    run()